# Module 7: Twist & Shout

## Mission Briefing

So far, your Pico has lived in a world of **on** and **off**. A button is pressed or it isn't. An LED is lit or it's dark.

Today, you discover the world **in between**. You'll connect a potentiometer — a twistable knob — and use it to control how bright an LED shines. Not just on. Not just off. *Any brightness you want.*

```
                           +--------------------+
                           |      Pico 2W       |
    +----------+  analog   |                    |  PWM output   +-----+
    |   KNOB   | -------->| GP26 (ADC0)        |              |     |
    | (potent) |  0-65535  |                    |------------>| LED |
    +----------+           |         GP15 (PWM) |  brightness  |     |
                           +--------------------+              +-----+
```

Turn the knob a little — the LED glows faintly. Crank it all the way — full blast. You're building a **dimmer switch**!

---
## Science Spotlight: Analog vs Digital

A button can only say **ON** or **OFF**. But what about "a little" or "a lot"?

A potentiometer is like a volume knob — it gives you every value in between. Instead of just 0 and 1, the Pico reads a number anywhere from **0 to 65,535**.

*Your instructor will explain how the Pico converts a smooth, real-world signal into a number it can understand — and why that number goes all the way up to 65,535.*

---
## Meet the Potentiometer

A potentiometer ("pot" for short) is a **variable resistor** with a twistable knob. It has **3 pins**:

```
        +-----------+
        |    KNOB   |
        |   (turn)  |
        +-----------+
         |    |    |
        Left Mid  Right
         |    |    |
       3.3V  |   GND
          (wiper)
             |
         To Pico
          GP26
```

- **Left pin** = power (3.3V)
- **Right pin** = ground (GND)
- **Middle pin** = the "wiper" — this is the output that changes as you turn the knob

When you twist the knob:
- All the way **left** = the middle pin outputs close to 3.3V (high number)
- All the way **right** = the middle pin outputs close to 0V (low number)
- **In between** = some voltage in between (some number in between)

**Tip:** The potentiometer from your Freenove kit is 10kOhm. Insert its 3 pins into 3 separate rows on the breadboard.

---
## Wiring: Potentiometer + LED

We'll wire both the potentiometer (input) and an LED (output) at the same time.

### Circuit

```
   Potentiometer Circuit:
   Pico 3.3V ──── wire ──── Pot Left Pin
   Pico GND  ──── wire ──── Pot Right Pin
   Pico GP26 ──── wire ──── Pot Middle Pin (wiper)

   LED Circuit (same as Module 2):
   Pico GP15 ──── [220 ohm Resistor] ──── LED (+) ──── LED (-) ──── GND
```

### Step-by-Step

1. **Place the potentiometer** on the breadboard — each of its 3 pins in a different row
2. **Connect a wire** from **3.3V** on the Pico to the **left pin** of the potentiometer
3. **Connect a wire** from **GND** on the Pico to the **right pin** of the potentiometer
4. **Connect a wire** from **GP26** on the Pico to the **middle pin** (wiper) of the potentiometer
5. **Add an LED** to the breadboard (long leg = +, short leg = -)
6. **Add a 220 ohm resistor** connecting GP15 to the LED's long leg row
7. **Connect** the LED's short leg row to GND

Plug in your USB cable.

---
## Code Along: Read the Potentiometer

Let's see what the potentiometer is telling the Pico. We use `ADC` (Analog-to-Digital Converter) to read the analog voltage.

Fill in the blank:

In [ ]:
from machine import ADC
import time

adc = ADC(_____)                # Which GPIO pin is the potentiometer wiper on?

for i in range(20):
    value = adc.read_u16()      # Read a 16-bit value (0 to 65535)
    print("Pot value:", value)
    time.sleep(0.5)

Run the code and **slowly twist the knob** while it prints values.

- What's the **smallest** number you can get? ____________
- What's the **largest** number you can get? ____________
- What happens when the knob is in the middle? ____________

You're reading an **analog** signal! The Pico turns a smooth voltage into a number.

---
## Code Along: Map Pot Value to LED Brightness

Now let's use the potentiometer to control how bright the LED shines. We'll use **PWM** (Pulse Width Modulation) — the same trick from Module 5 that lets us set brightness levels.

The potentiometer gives us 0-65535. PWM duty cycle also takes 0-65535. That's a perfect match!

Fill in the blanks:

In [ ]:
from machine import Pin, ADC, PWM
import time

adc = ADC(_____)                    # Potentiometer pin
led_pwm = PWM(Pin(_____))           # LED pin
led_pwm.freq(1000)

while True:
    pot_value = adc._____()         # What method reads the value?
    led_pwm.duty_u16(_____)         # What value controls brightness?
    print("Brightness:", pot_value)
    time.sleep(0.05)

Turn the knob slowly. The LED should get brighter and dimmer as you twist!

You just built a **dimmer switch** — the same concept used in real room lighting controls.

---
## Code Along: Mapping Values

Sometimes the input range and output range don't match. What if you wanted to map the potentiometer's 0-65535 to a percentage (0-100)?

The formula is:
```
mapped_value = output_min + (value - input_min) * (output_max - output_min) // (input_max - input_min)
```

Let's write a helper function. Fill in the blanks:

In [ ]:
from machine import ADC
import time

adc = ADC(26)

def map_range(value, in_min, in_max, out_min, out_max):
    return out_min + (value - in_min) * (_____ - _____) // (in_max - in_min)

for i in range(20):
    pot_value = adc.read_u16()
    percentage = map_range(pot_value, 0, _____, 0, _____)   # Map to 0-100
    print("Knob is at", percentage, "%")
    time.sleep(0.5)

Now instead of a confusing number like 32768, you see a friendly percentage like 50%!

Mapping values is a super useful skill. Any time you have a number in one range and need it in another, this function does the job.

---
## Experiments

Try these modifications:

1. **Threshold alarm:** Modify the dimmer code so the LED only turns on when the potentiometer is past the halfway point. (Hint: check if `pot_value > 32767`)

2. **Flip it:** Can you make the LED **brightest** when the knob is turned all the way down? (Hint: `65535 - pot_value`)

3. **Speed control:** Instead of brightness, use the potentiometer to control how fast the LED blinks. Map the pot value to a `time.sleep()` delay.

4. **Range check:** Use the `map_range` function to map the pot value to a range of 1-10. Print which "level" you're at.

---
## Challenge: Mood Light

If you still have the RGB LED circuit from Module 6, try this:

**Build a "mood light"** where the potentiometer controls the color of the RGB LED!

### Ideas:
- Divide the pot range into zones: low = red, middle = green, high = blue
- Or smoothly blend between colors as you turn the knob
- Or control just the red channel with the pot and keep green and blue fixed

### Useful hints:

```python
# To divide into 3 zones:
if pot_value < 21845:
    # Zone 1: red
elif pot_value < 43690:
    # Zone 2: green
else:
    # Zone 3: blue

# Remember: 65535 / 3 = ~21845
```

Ask your instructor for help if you get stuck!

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Analog** | A smooth, continuous signal — like a volume knob that can be anywhere between min and max |
| **Digital** | A signal with only distinct levels — like a light switch that's either ON or OFF |
| **ADC (Analog-to-Digital Converter)** | Hardware in the Pico that turns an analog voltage into a digital number |
| **Potentiometer** | A variable resistor with a twistable knob — the middle "wiper" pin outputs a changing voltage |
| **Range mapping** | Converting a number from one range (e.g. 0-65535) to another range (e.g. 0-100) |
| **Resolution** | How many different values the ADC can distinguish — the Pico uses 16-bit (65,536 levels) |
| **PWM (Pulse Width Modulation)** | Rapidly switching a pin on and off to simulate in-between values (like LED brightness) |

---
*Circuit Explorers — Module 7: Twist & Shout*